# 03 — Tool Use & Function Calling

## What is Function Calling?

Function calling (also called **tool use**) lets Gemini Live do more than just talk — it lets the model **invoke real code** in your application mid-conversation.

Instead of making up answers, the model can:
- Call your Python functions to get live data
- Chain multiple tool calls together
- Combine tool results to give a complete, factual answer

### Why does this matter for agents?

| Without tools | With tools |
|---|---|
| Model hallucinates answers | Model calls real APIs |
| Static knowledge cutoff | Live, dynamic data |
| Can't act on the world | Can trigger actions (send email, query DB) |
| Single-turn answers | Multi-step reasoning over real state |

### How it works in Gemini Live:

1. You **declare** tools in `LiveConnectConfig`
2. User speaks a request
3. Model decides it needs a tool → sends a `tool_call` event
4. Your code **executes** the function locally
5. You send results back via `send_tool_response`
6. Model incorporates results into its spoken reply

This notebook covers three progressively complex demos: calculator, weather, and multi-tool.

## Setup

Install dependencies and configure the client.

In [ ]:
# Install the Google GenAI SDK if not already present
# !pip install -q google-genai numpy

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.run() inside Jupyter

import asyncio
import os
import json
import datetime
import numpy as np
import IPython.display as ipd
from google import genai
from google.genai import types

# --- Configuration ---
from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"

client = genai.Client(api_key=API_KEY)

print(f"SDK ready. Model: {MODEL}")

## Helper: PCM Audio Utilities

These helpers let us synthesize test audio and play back model responses.

In [ ]:
def make_pcm(duration=2.0, rate=16000, freq=440):
    """Generate a sine-wave PCM clip at the given frequency."""
    t = np.linspace(0, duration, int(rate * duration))
    return (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16).tobytes()


def play_pcm(raw_bytes, rate=24000):
    """Return an IPython Audio widget from raw PCM bytes (int16)."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


print("Audio helpers ready.")

---
## Demo 1 — Simple Calculator Tool

We define a Python `calculator` function that handles add/subtract/multiply/divide,
declare it as a `FunctionDeclaration`, and let the model call it.

### Step 1: Define the Python function

In [ ]:
def get_calculator(operation: str, a: float, b: float) -> dict:
    """
    Perform a basic arithmetic operation.

    Args:
        operation: One of 'add', 'subtract', 'multiply', 'divide'
        a: First operand
        b: Second operand

    Returns:
        dict with 'result' key, or 'error' if invalid input
    """
    ops = {
        "add":      a + b,
        "subtract": a - b,
        "multiply": a * b,
    }
    if operation == "divide":
        if b == 0:
            return {"error": "Cannot divide by zero"}
        return {"result": a / b}
    if operation not in ops:
        return {"error": f"Unknown operation: {operation}"}
    return {"result": ops[operation]}


# Quick sanity check
print(get_calculator("multiply", 347, 28))  # → {'result': 9716}

### Step 2: Declare the tool schema

The model needs a JSON Schema description so it knows when and how to call the function.

In [ ]:
calculator_declaration = types.FunctionDeclaration(
    name="get_calculator",
    description="Perform basic arithmetic: add, subtract, multiply, or divide two numbers.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "operation": types.Schema(
                type=types.Type.STRING,
                description="The arithmetic operation: 'add', 'subtract', 'multiply', or 'divide'.",
                enum=["add", "subtract", "multiply", "divide"],
            ),
            "a": types.Schema(
                type=types.Type.NUMBER,
                description="The first operand.",
            ),
            "b": types.Schema(
                type=types.Type.NUMBER,
                description="The second operand.",
            ),
        },
        required=["operation", "a", "b"],
    ),
)

calculator_tools = [types.Tool(function_declarations=[calculator_declaration])]

print("Tool schema declared.")

### Step 3: Run the session

In [ ]:
async def demo_calculator():
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        tools=calculator_tools,
        system_instruction="You are a helpful math assistant. Use the calculator tool for all arithmetic.",
    )

    query = "What is 347 multiplied by 28?"
    print(f"User: {query}\n")

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=query)

        full_text = ""

        async for resp in session.receive():
            # --- Handle tool call ---
            if resp.tool_call:
                for fc in resp.tool_call.function_calls:
                    args = dict(fc.args)
                    print(f"  [Tool call] {fc.name}({args})")
                    result = get_calculator(**args)
                    print(f"  [Tool result] {result}")
                    await session.send_tool_response(
                        function_responses=[
                            types.FunctionResponse(
                                id=fc.id,
                                name=fc.name,
                                response=result,
                            )
                        ]
                    )

            # --- Collect transcript from audio response ---
            if resp.server_content:
                sc = resp.server_content
                if sc.output_transcription and sc.output_transcription.text:
                    full_text += sc.output_transcription.text
                if sc.turn_complete:
                    break

            if resp.go_away:
                print("Session ended by server.")
                break

    print(f"\nModel: {full_text}")


asyncio.run(demo_calculator())


---
## Demo 2 — Weather Tool (Mock Data)

This demo introduces:
- A more realistic tool shape (city lookup → dict with multiple fields)
- Handling **multiple tool calls in a single turn** (model asks for both cities at once)

### Step 1: Define the mock weather function

In [ ]:
# Mock weather database — replace with a real API call in production
WEATHER_DB = {
    "Singapore": {"temperature_c": 31, "condition": "Sunny",  "humidity_pct": 80, "wind_kmh": 12},
    "Tokyo":     {"temperature_c": 18, "condition": "Cloudy", "humidity_pct": 55, "wind_kmh": 20},
    "London":    {"temperature_c": 12, "condition": "Rainy",  "humidity_pct": 85, "wind_kmh": 30},
    "New York":  {"temperature_c": 22, "condition": "Clear",  "humidity_pct": 60, "wind_kmh": 15},
    "Sydney":    {"temperature_c": 25, "condition": "Partly Cloudy", "humidity_pct": 65, "wind_kmh": 18},
}


def get_weather(city: str) -> dict:
    """
    Return current weather for a city.

    Args:
        city: Name of the city (case-sensitive, matches DB keys)

    Returns:
        dict with weather details or an error message
    """
    data = WEATHER_DB.get(city)
    if data is None:
        return {"error": f"No weather data for '{city}'. Available: {list(WEATHER_DB.keys())}"}
    return {"city": city, **data}


print(get_weather("Singapore"))

### Step 2: Declare the weather tool

In [ ]:
weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get current weather conditions for a specified city.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "city": types.Schema(
                type=types.Type.STRING,
                description="The city name to look up weather for (e.g. 'Singapore', 'Tokyo').",
            ),
        },
        required=["city"],
    ),
)

weather_tools = [types.Tool(function_declarations=[weather_declaration])]
print("Weather tool declared.")

### Step 3: Run the session — note how we handle multiple tool calls

In [ ]:
async def demo_weather():
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        tools=weather_tools,
        system_instruction="You are a weather assistant. Always use the get_weather tool for current conditions.",
    )

    query = "What's the weather in Singapore and Tokyo?"
    print(f"User: {query}\n")

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=query)

        full_text = ""

        async for resp in session.receive():
            if resp.tool_call:
                function_responses = []
                for fc in resp.tool_call.function_calls:
                    args = dict(fc.args)
                    print(f"  [Tool call] {fc.name}({args})")
                    if fc.name == "get_weather":
                        result = get_weather(**args)
                    else:
                        result = {"error": f"Unknown tool: {fc.name}"}
                    print(f"  [Tool result] {result}")
                    function_responses.append(
                        types.FunctionResponse(id=fc.id, name=fc.name, response=result)
                    )
                await session.send_tool_response(function_responses=function_responses)

            if resp.server_content:
                sc = resp.server_content
                if sc.output_transcription and sc.output_transcription.text:
                    full_text += sc.output_transcription.text
                if sc.turn_complete:
                    break

            if resp.go_away:
                break

    print(f"\nModel: {full_text}")


asyncio.run(demo_weather())


---
## Demo 3 — Multi-Tool Session

Now we combine **calculator + weather + time** in a single session.
The model picks the right tool(s) based on what the user asks.

### Step 1: Add the time tool

In [ ]:
def get_current_time(timezone: str = "UTC") -> dict:
    """
    Return the current date and time.

    Args:
        timezone: Timezone string (e.g., 'UTC', 'Asia/Singapore')
                  Note: this mock always returns UTC for simplicity.

    Returns:
        dict with 'datetime', 'date', 'time', and 'timezone' keys
    """
    now = datetime.datetime.utcnow()
    return {
        "datetime": now.isoformat(),
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "timezone": timezone,
        "note": "This mock always returns UTC regardless of timezone.",
    }


print(get_current_time("Asia/Singapore"))

### Step 2: Declare all three tools together

In [ ]:
time_declaration = types.FunctionDeclaration(
    name="get_current_time",
    description="Get the current date and time, optionally for a specific timezone.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "timezone": types.Schema(
                type=types.Type.STRING,
                description="IANA timezone string, e.g. 'UTC', 'Asia/Singapore', 'America/New_York'.",
            ),
        },
        required=[],  # timezone is optional
    ),
)

# Bundle all three declarations into a single Tool object
all_tools = [
    types.Tool(
        function_declarations=[
            calculator_declaration,
            weather_declaration,
            time_declaration,
        ]
    )
]

print(f"Tools registered: {[d.name for d in all_tools[0].function_declarations]}")

### Step 3: Tool dispatcher

A clean dispatcher function maps tool names to local implementations.

In [ ]:
TOOL_REGISTRY = {
    "get_calculator":  get_calculator,
    "get_weather":     get_weather,
    "get_current_time": get_current_time,
}


def dispatch_tool(name: str, args: dict) -> dict:
    """Route a tool call to the matching Python function."""
    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        return {"error": f"Tool not found: {name}"}
    try:
        return fn(**args)
    except Exception as e:
        return {"error": str(e)}


print("Dispatcher ready.")

### Step 4: Run the multi-tool session

In [ ]:
async def demo_multi_tool(query: str):
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        tools=all_tools,
        system_instruction=(
            "You are a helpful assistant with access to a calculator, weather data, and a clock. "
            "Use the appropriate tool(s) to answer the user's question accurately."
        ),
    )

    print(f"User: {query}\n")

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        await session.send_realtime_input(text=query)

        full_text = ""

        async for resp in session.receive():
            if resp.tool_call:
                function_responses = []
                for fc in resp.tool_call.function_calls:
                    args = dict(fc.args)
                    print(f"  [Tool call] {fc.name}({args})")
                    result = dispatch_tool(fc.name, args)
                    print(f"  [Result]    {result}")
                    function_responses.append(
                        types.FunctionResponse(id=fc.id, name=fc.name, response=result)
                    )
                await session.send_tool_response(function_responses=function_responses)

            if resp.server_content:
                sc = resp.server_content
                if sc.output_transcription and sc.output_transcription.text:
                    full_text += sc.output_transcription.text
                if sc.turn_complete:
                    break

            if resp.go_away:
                break

    print(f"\nModel: {full_text}")
    return full_text


asyncio.run(demo_multi_tool(
    "What time is it now, what's the weather in London, and what is 1440 divided by 24?"
))


In [ ]:
# Try another query — this one only triggers weather + calculator
asyncio.run(demo_multi_tool(
    "Is it warmer in Singapore or Tokyo? And by how many degrees Celsius?"
))

---
## Tool Declaration Pattern — Schema Reference

Here is the full pattern for declaring tools with different schema types:

```python
types.FunctionDeclaration(
    name="my_function",
    description="What this tool does — be specific, the model uses this to decide when to call it.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            # String parameter
            "city": types.Schema(
                type=types.Type.STRING,
                description="A city name",
            ),
            # Number parameter
            "count": types.Schema(
                type=types.Type.NUMBER,
                description="A numeric count",
            ),
            # Boolean parameter
            "include_details": types.Schema(
                type=types.Type.BOOLEAN,
                description="Whether to include full details",
            ),
            # Enum (constrained string)
            "operation": types.Schema(
                type=types.Type.STRING,
                enum=["add", "subtract"],
                description="The operation to perform",
            ),
            # Array of strings
            "tags": types.Schema(
                type=types.Type.ARRAY,
                items=types.Schema(type=types.Type.STRING),
                description="A list of tag strings",
            ),
        },
        required=["city", "operation"],  # Only list truly required params
    ),
)
```

### Schema types available:

| Type | Use for |
|------|--------|
| `types.Type.STRING` | Text, city names, identifiers |
| `types.Type.NUMBER` | Floats and integers |
| `types.Type.INTEGER` | Whole numbers only |
| `types.Type.BOOLEAN` | True/false flags |
| `types.Type.ARRAY` | Lists (pair with `items=`) |
| `types.Type.OBJECT` | Nested objects |

### Pro tips:
- Write clear `description` strings — the model uses them to decide **when** to call the tool
- Use `enum` to constrain string options (prevents hallucinated values)
- Only put truly required params in `required=[]` — optional params give the model flexibility
- Return a `dict` — it gets serialized as JSON back to the model

---
## Key Takeaways

1. **Declare tools in `LiveConnectConfig`** — the model sees them before the session starts

2. **The receive loop handles tool calls** — check `resp.tool_call` before checking `resp.server_content`

3. **Send results with `send_tool_response`** — using `FunctionResponse(id=fc.id, name=fc.name, response=dict)`

4. **Batch multiple responses** — if the model calls 3 tools at once, send all 3 `FunctionResponse` objects in one `send_tool_response` call

5. **Use a dispatcher pattern** — a `TOOL_REGISTRY` dict mapping names to functions keeps your receive loop clean

6. **NEVER mix `send_client_content` with `send_realtime_input`** — use only `send_realtime_input` for text, audio, and tool results

### What to build next:
- Replace mock functions with real API calls (OpenWeatherMap, Google Maps, etc.)
- Add a database query tool for RAG
- Chain tool results: weather → clothing recommendation → calendar booking
- Combine with audio input so users can *speak* their requests